In [ ]:
# ==========================================
# PROYECTO: EXTRACCIÓN DE DATOS URBANOS
# COMUNA: San Miguel, Santiago, Chile
# Autor: Enfoque Data Science + Ingeniería Urbana
# ==========================================

# 1. Instalación (ejecutar una vez)
!pip install osmnx geopandas shapely fiona pyproj

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.8 MB/s eta 0:00:00


In [ ]:
# 2. Librerías
import osmnx as ox
import geopandas as gpd
import os

In [ ]:
# 3. Configuración
ox.settings.log_console = True
ox.settings.use_cache = True

# Carpeta de salida
output_dir = "san_miguel_capas"
os.makedirs(output_dir, exist_ok=True)

# Área de estudio
place_name = "San Miguel, Santiago, Chile"

# Sistema de coordenadas proyectado (Chile)
CRS_PROY = "EPSG:32719"

print("Descargando datos para:", place_name)

Descargando datos para: San Miguel, Santiago, Chile


In [ ]:
# ==========================================
# 4. RED VIAL (GRAFO)
# ==========================================
G = ox.graph_from_place(place_name, network_type="drive")
nodes, edges = ox.graph_to_gdfs(G)

edges = edges.to_crs(CRS_PROY)
nodes = nodes.to_crs(CRS_PROY)

edges.to_file(f"{output_dir}/red_vial.shp")
nodes.to_file(f"{output_dir}/nodos_viales.shp")

print("✔ Red vial descargada")

✔ Red vial descargada


/tmp/ipykernel_12500/3843960651.py:11: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  nodes.to_file(f"{output_dir}/nodos_viales.shp")
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'street_count' to 'street_cou'
  ogr_write(


In [ ]:
# ==========================================
# 5. NODOS CRÍTICOS (INTERSECCIONES)
# ==========================================
nodos_criticos = nodes[nodes["street_count"] >= 4]
nodos_criticos.to_file(f"{output_dir}/nodos_criticos.shp")

print("✔ Nodos críticos identificados")

✔ Nodos críticos identificados


/tmp/ipykernel_12500/4256245820.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  nodos_criticos.to_file(f"{output_dir}/nodos_criticos.shp")
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'street_count' to 'street_cou'
  ogr_write(


In [ ]:
# ==========================================
# 6. EQUIPAMIENTO URBANO (POIs)
# ==========================================
tags_pois = {
    "amenity": True,
    "shop": True,
    "tourism": True
}

pois = ox.features_from_place(place_name, tags_pois)

if not pois.empty:
    pois = pois.to_crs(CRS_PROY) # Reproject all features once

    # Separate geometries by type
    pois_points = pois[pois.geometry.geom_type.isin(['Point', 'MultiPoint'])]
    pois_polygons = pois[pois.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]

    saved_types = []

    if not pois_points.empty:
        pois_points.to_file(f"{output_dir}/equipamiento_urbano_points.shp")
        saved_types.append("puntos")

    if not pois_polygons.empty:
        pois_polygons.to_file(f"{output_dir}/equipamiento_urbano_polygons.shp")
        saved_types.append("polígonos")

    if saved_types:
        print(f"✔ POIs descargados ({', '.join(saved_types)} guardados en archivos separados)")
    else:
        print("⚠ No se encontraron POIs con geometrías válidas para guardar.")
else:
    print("⚠ No se encontraron POIs")

✔ POIs descargados (puntos, polígonos guardados en archivos separados)


/tmp/ipykernel_12500/3700173241.py:22: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  pois_points.to_file(f"{output_dir}/equipamiento_urbano_points.shp")
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:city' to 'addr_city'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:full' to 'addr_full'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:housenumber' to 'addr_house'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:street' to 'addr_stree'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'brand:wikidata' to 'brand_wiki'
  ogr_write(
/usr/local/lib/python3.12/dis

In [ ]:
# ==========================================
# 7. PARADEROS DE BUS
# ==========================================
tags_bus = {"highway": "bus_stop"}
bus_stops = ox.features_from_place(place_name, tags_bus)

if not bus_stops.empty:
    bus_stops = bus_stops.to_crs(CRS_PROY)
    bus_stops.to_file(f"{output_dir}/paraderos.shp")
    print("✔ Paraderos descargados")

✔ Paraderos descargados


/tmp/ipykernel_12500/2117624835.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  bus_stops.to_file(f"{output_dir}/paraderos.shp")
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'attribution' to 'attributio'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'is_in:city' to 'is_in_city'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'network:wikidata' to 'network_wi'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'public_transport' to 'public_tra'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'ts_orientacion' to 'ts_orienta'
  ogr_write(
/usr/local/lib/python3.12/dist-packages

In [ ]:
# ==========================================
# 8. INFRAESTRUCTURA PEATONAL
# ==========================================
tags_walk = {"highway": ["footway", "pedestrian", "path"]}
walkways = ox.features_from_place(place_name, tags_walk)

if not walkways.empty:
    walkways = walkways.to_crs(CRS_PROY)
    walkways.to_file(f"{output_dir}/infraestructura_peatonal.shp")
    print("✔ Infraestructura peatonal descargada")

✔ Infraestructura peatonal descargada


/tmp/ipykernel_12500/589490865.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  walkways.to_file(f"{output_dir}/infraestructura_peatonal.shp")
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'crossing:markings' to 'crossing_m'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'motor_vehicle' to 'motor_vehi'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'crossing:island' to 'crossing_i'
  ogr_write(


In [ ]:
# ==========================================
# 9. CICLOVÍAS
# ==========================================
tags_cycle = {"highway": "cycleway"}
cycleways = ox.features_from_place(place_name, tags_cycle)

if not cycleways.empty:
    cycleways = cycleways.to_crs(CRS_PROY)
    cycleways.to_file(f"{output_dir}/ciclovias.shp")
    print("✔ Ciclovías descargadas")
else:
    print("⚠ No se encontraron ciclovías con la etiqueta 'highway=cycleway'.")

✔ Ciclovías descargadas


In [ ]:
# ==========================================
# 10. USO DE SUELO
# ==========================================
tags_landuse = {"landuse": True}
landuse = ox.features_from_place(place_name, tags_landuse)

if not landuse.empty:
    landuse = landuse.to_crs(CRS_PROY)
    landuse.to_file(f"{output_dir}/uso_suelo.shp")
    print("✔ Uso de suelo descargado")

✔ Uso de suelo descargado


/tmp/ipykernel_12500/1484894511.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  landuse.to_file(f"{output_dir}/uso_suelo.shp")
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:city' to 'addr_city'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:housenumber' to 'addr_house'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:street' to 'addr_stree'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'construction' to 'constructi'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'description' to 'descriptio'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/ra

In [ ]:
# ==========================================
# 11. LIMPIEZA BÁSICA (OPCIONAL)
# ==========================================
# Eliminar geometrías inválidas
def limpiar_gdf(gdf):
    return gdf[gdf.geometry.notnull()]

In [ ]:
# ==========================================
# 12. RESUMEN FINAL
# ==========================================
print("\\n===== RESUMEN =====")
print(f"Nodos totales: {len(nodes)}")
print(f"Calles totales: {len(edges)}")
print(f"Nodos críticos: {len(nodos_criticos)}")

print("\\nArchivos guardados en carpeta:", output_dir)
print("Proceso completado correctamente")

\n===== RESUMEN =====
Nodos totales: 1171
Calles totales: 2312
Nodos críticos: 438
\nArchivos guardados en carpeta: san_miguel_capas
Proceso completado correctamente
